# Kaggle runner: test evaluation, ablations, example images
Settings: GPU on, Internet on. Inputs: dataset `nih-chest-xrays/data` and your private model `best-model`.
Run the sections top to bottom. Nothing here edits the pipeline: every experiment goes through `src.train` / `src.evaluate`.

## 0. Setup

In [ ]:
!git clone https://github.com/khldd/chest-xray-diagnosis.git repo
%cd repo
!pip install -q -r requirements/train.txt
import torch; print("CUDA:", torch.cuda.is_available())

In [ ]:
# Find the uploaded weights (the path depends on how Kaggle mounted the model)
import glob
W = glob.glob("/kaggle/input/**/best_model.pth", recursive=True)
print(W)
WEIGHTS = W[0]

## 1. Faithfulness check: validation AUC must be about 0.831

In [ ]:
!python -m src.evaluate --weights {WEIGHTS} --split val

## 2. Official test set (commit this JSON as `results/test_metrics.json`)

In [ ]:
!mkdir -p results
!python -m src.evaluate --weights {WEIGHTS} --split test --out results/test_metrics.json

## 3. Ablations (5 epochs each, roughly 2 h per run on a T4). Run one per cell/session to protect the quota.

In [ ]:
!python -m src.train --config configs/base_5ep.yaml --run-name base_5ep

In [ ]:
!python -m src.train --config configs/baseline_frozen.yaml --run-name baseline_frozen

In [ ]:
!python -m src.train --config configs/unweighted_bce.yaml --run-name unweighted_bce

In [ ]:
!python -m src.train --config configs/strong_aug.yaml --run-name strong_aug

## 4. Export results table (commit as `results/ablations.csv`)

In [ ]:
import mlflow, pandas as pd
runs = mlflow.search_runs(experiment_names=["chest-xray-ablations"])
cols = ["tags.mlflow.runName", "params.model.freeze_backbone", "params.train.loss",
        "params.data.augmentation", "params.train.epochs", "metrics.best_val_auc", "metrics.val_mean_auc"]
cols += sorted(c for c in runs.columns if c.startswith("metrics.auc_"))
table = runs[cols].rename(columns={"tags.mlflow.runName": "run"}).sort_values("run")
table.to_csv("results/ablations.csv", index=False)
table

## 5. Example X-rays for the frontend gallery (test split only)

In [ ]:
import os, json, pandas as pd
from PIL import Image
from src.data.prepare import make_splits, build_image_lookup
ROOT = "/kaggle/input/datasets/nih-chest-xrays/data"
test_df, test_y = make_splits(ROOT)["test"]
lookup = build_image_lookup(ROOT)
# Indices picked earlier in the notebook (test split ordering is identical): add 5-6 more of your choice
PICKS = {278: "Cardiomegaly", 197: "Infiltration"}
os.makedirs("examples", exist_ok=True); manifest = []
for idx, label in PICKS.items():
    name = test_df.loc[idx, "Image Index"]
    img = Image.open(lookup[name]).convert("L").resize((512, 512))   # keeps each file well under 1 MB
    img.save(f"examples/{name}"); manifest.append({"file": name, "label": label})
json.dump(manifest, open("examples/manifest.json", "w"), indent=2)
print(manifest)

Download from the Output panel: `results/test_metrics.json`, `results/ablations.csv`, `examples/`. Then commit them from your local clone, not from Kaggle.